**`enrich_parcels_zonal_stats`**

Attaches raster-derived statistics to a harmonized parcel spine.

The work is done by the recipe's own pipeline steps, not by this notebook:

- `zonal_stats` samples a raster over each parcel polygon.

- `vicinity_coverage` first derives a neighborhood-percentage raster, then samples that.

- `derive_from_spine` adds area and density metrics that need no raster.

Rasters are named in the recipe by a path relative to the configured `rasters`
directory, so the same recipe runs anywhere.


# Configure

In [ ]:
import argparse

import openplaces as op
from openplaces.flow import convert_to_script
from openplaces.io.enricher import enrich

In [ ]:
parser = argparse.ArgumentParser(
    description='Enrich a parcel spine with raster-derived statistics'
)
parser.add_argument(
    '--recipe_id',
    default='CO_parcel_land-zonal-stats-multi-2026',
    help='Enrichment recipe to run',
)
parser.add_argument(
    '--entity_recipe_id',
    default='CO_parcel-spine-2026',
    help='Harmonized parcel spine the enrichment reads',
)
parser.add_argument(
    '--admin_ids',
    nargs='*',
    help='Admin unit IDs to process (e.g. "CO-CH")',
)
parser.add_argument(
    '--reprocess',
    action='store_true',
    help='Rerun admin units that already have output',
)
parser.add_argument(
    '--verbose',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    '--recipe_id CO_parcel_land-zonal-stats-multi-2026 '
    '--entity_recipe_id CO_parcel-spine-2026 '
    '--admin_ids CO-CH '
    # '--reprocess '
    '--verbose '
)

args_list = [x for x in ARGS_TEST.split(' ') if x != '']
args = parser.parse_args(args_list)
args

# Enrich

Each admin unit is processed independently, so a unit with no parcel coverage
is skipped rather than failing the run.


In [ ]:
enrich(
    args.recipe_id,
    admin_ids=args.admin_ids,
    entity_recipe_id=args.entity_recipe_id,
    reprocess=args.reprocess,
    verbose=args.verbose,
)

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*


In [ ]:
COMMIT = False
# If True, writes `.py` scripts to 'scripts/'.
# If False, writes a test version of the script to 'scripts/_test/'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
# test_script(*args_list, committed=COMMIT)


# Inspect results

In [ ]:
import pandas as pd

evidence = op.get_entities(args.recipe_id, args.admin_ids[0])
pd.options.display.max_rows = evidence.shape[1] + 5
print(len(evidence))
evidence.sample(5).T

In [ ]:
# Coverage of each derived column, to spot a raster that produced nothing
(evidence.notna().mean() * 100).round(1).sort_values(ascending=False).head(30)